In [ ]:
import torch
import pandas as pd
import numpy as np

PT_PATH = "C:/Users/user/Desktop/ids_masters/small_CAN_MIRGU/training_dataset.pt"

feature_names = [
    "Global IAT", "ID IAT", "Entropy", "Hamming", "DLC",
    "Delta Payload", "Jitter", "Byte Mean", "Byte Std"
]
ATTACK_LABELS = {
    "normal": 0,
    "dos": 1,
    "fuzzing": 2,
    "replay": 3,
    "spoofing": 4,
}
label_map = {v: k for k, v in ATTACK_LABELS.items()}

# 1) load
data = torch.load(PT_PATH, map_location="cpu")
X = data["X"].numpy()   # (N, 64, 9)
y = data["y"].numpy()   # (N,)
X = np.transpose(X, (0, 2, 1))

N, T, F = X.shape
assert T == 64 and F == 9, (X.shape, "expected (N,64,9)")

# 2) make pandas table: (N*64, 9)
df_all = pd.DataFrame(X.reshape(N*T, F), columns=feature_names)

# window index / timestep
df_all.insert(0, "t", np.tile(np.arange(T), N))
df_all.insert(0, "window_idx", np.repeat(np.arange(N), T))

# window label (repeated for each of 64 rows)
df_all["window_y"] = y[df_all["window_idx"].values].astype(int)
df_all["window_label"] = df_all["window_y"].map(label_map)

# 3) save as parquet
try:
    CSV_PATH = "C:/Users/user/Desktop/ids_masters/small_CAN_MIRGU/windows_all.csv"

    df_all.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    print(" saved csv:", CSV_PATH)

except Exception as e:
    print(" csv 저장 실패:", repr(e))
    print(" 해결: `pip install pyarrow` 또는 `pip install fastparquet` 후 다시 실행")
